In [1]:
from run_easg import EASGData
from dataset import myEASGDataset
from pathlib import Path
with open('annts_in_new_format/' + 'verbs.txt') as f:
    verbs = [l.strip() for l in f.readlines()]
num_verbs = len(verbs)

with open('annts_in_new_format/' + 'objects.txt') as f:
    objs = [l.strip() for l in f.readlines()]
num_objs = len(objs)

with open('annts_in_new_format/' + 'relationships.txt') as f:
    rels = [l.strip() for l in f.readlines()]
num_rels = len(rels)

path_annts = Path('annts_in_new_format')
path_data = Path('data')

train_original = EASGData(path_annts, path_data, 'train', verbs, objs, rels)
train_dataset = myEASGDataset(train_original)
batch_size = 4

# Create ground truth

In [4]:
train_dataset.get_original_triplets(0)

tensor([[162, 212,   1],
        [162, 171,   3]])

In [6]:
train_dataset[0].y[1][170]

tensor([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [3]:
import torch
ground_truth = torch.zeros((391,14))
print(ground_truth.size())
ground_truth[:,13]=1
for el in train_dataset.get_original_triplets(2):
    ground_truth[el[1]-1,el[2]-1] = 1
    ground_truth[el[1]-1,13] = 0

torch.Size([391, 14])


In [44]:
ground_truth

tensor([[0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 1.]])

# Check if training with batches makes sense

In [2]:
from torch_geometric.loader import DataLoader
import torch
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

In [3]:
from autoencoder import LinearProjection, myGNN, MyAutoEncoder

lp = LinearProjection(2304, 1024, 512, 512, 'cpu', 0.5)
gnn = myGNN('gcn', 512, 256, 64, 'add', 0.5, 'cpu')


In [4]:
autoenc = MyAutoEncoder(1204, 2304, 14, 198, 391, 512, 256, 128, 64, 128, 2, 6, 'add', 'cuda', 0.2, 'gin')

In [5]:
for batch in train_loader:
    out = lp(batch)
    print(out)
    break

DataBatch(x=[12, 512], edge_index=[2, 8], y=[2], batch=[12], ptr=[5])


In [6]:
out[0]

Data(x=[3, 512], edge_index=[2, 2], y=[2])

# See what their autoencoder does

In [2]:
from autoencoder import Decoder
import torch

In [3]:
latent_dim = 64
hidden_dim = 128
n_layers = 2
n_nodes = 3
dec = Decoder(latent_dim, hidden_dim, n_layers, n_nodes)

In [4]:
t = torch.rand((1,64))
dec(t)

tensor([[[0., 0., 0.],
         [0., 0., 1.],
         [0., 1., 0.]]], grad_fn=<AddBackward0>)